# 👁️ Module 04: Spatial Vision & Convolutions (CNNs) (1998 - 2022)
### *From LeNet-5 & Receptive Fields to the ResNet Revolution & ConvNeXt*

---

## 📜 Historical Context & The Three Spatial Frontiers
In 1998, **Yann LeCun** and colleagues published the landmark paper introducing **LeNet-5**, demonstrating that machines could read handwritten digits with superhuman accuracy.

Why did dense neural networks fail on images? Why was the Convolutional Neural Network (CNN) necessary?
1. **The Curse of Dimensionality & Spatial Blindness**: An MLP treats pixel (0,0) and pixel (0,1) as having no more relationship than pixel (0,0) and pixel (224,224).
2. **The Receptive Field & Spatial Hierarchy**: How local features (edges, corners) compose into complex semantics (eyes, wheels, faces).
3. **The 2015 Deep Degradation Crisis**: Kaiming He et al. discovered that stacking 50+ plain layers causes accuracy to collapse — resolved by **ResNet's Identity Shortcut Connection**.
4. **ConvNeXt (2022)**: Modernizing CNNs with Vision Transformer principles.

## 1. Spatial Locality & The Stenciled Flashlight Analogy

### 🔦 The Metaphor:
> Imagine examining a massive painting in a pitch-black gallery. An MLP turns on a colossal floodlight connecting every microscopic pigment particle to every receptor in your eye simultaneously — requiring hundreds of millions of parameters.
> A Convolutional Layer instead hands you a **stenciled flashlight** (the kernel). You systematically sweep it across the canvas (stride), scanning for localized visual patterns regardless of where they sit in space (**translation equivariance**).

### Mathematical Definition of 2D Discrete Convolution:
$$S(i, j) = (I * K)(i, j) = \sum_{m} \sum_{n} I(i + m, j + n) K(m, n) + b$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a synthetic 16x16 test pattern with vertical and horizontal edges
image = np.zeros((16, 16))
image[4:12, 4:12] = 1.0
image[7:9, 7:9] = 0.5

# Define classic spatial kernels
sobel_vertical = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float64)

sobel_horizontal = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float64)

sharpen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float64)

def apply_kernel_2d(img, kernel):
    kh, kw = kernel.shape
    h, w = img.shape
    out = np.zeros((h - kh + 1, w - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(img[i:i+kh, j:j+kw] * kernel)
    return out

out_v = apply_kernel_2d(image, sobel_vertical)
out_h = apply_kernel_2d(image, sobel_horizontal)
out_s = apply_kernel_2d(image, sharpen)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(image, cmap='magma'); axes[0].set_title('Original Image (16x16)')
axes[1].imshow(out_v, cmap='coolwarm'); axes[1].set_title('Sobel Vertical Edges')
axes[2].imshow(out_h, cmap='coolwarm'); axes[2].set_title('Sobel Horizontal Edges')
axes[3].imshow(out_s, cmap='viridis'); axes[3].set_title('Sharpen Filter')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## 2. The Receptive Field Arithmetic

The **Receptive Field (RF)** of a unit in layer $l$ is the region of the original input canvas that influences its activation.

### Exact Recurrence Formulas:
For layer $l$ with kernel size $k_l$, stride $s_l$, padding $p_l$, and dilation $d_l$:
1. **Jump**: $j_l = j_{l-1} \cdot s_l$
2. **Receptive Field**: $r_l = r_{l-1} + (k_l - 1) \cdot d_l \cdot j_{l-1}$
3. **Center Coordinate**: $\text{start}_l = \text{start}_{l-1} + \left(\frac{k_l - 1}{2} \cdot d_l - p_l\right) \cdot j_{l-1}$

### The VGG Insight (Simonyan & Zisserman, 2014):
Why stack two $3 \times 3$ convolutions instead of one $5 \times 5$ convolution?
- Both cover an identical receptive field of $5 \times 5$.
- But two $3 \times 3$ filters use $2 \times (3^2 \cdot C^2) = 18 C^2$ parameters vs $5^2 \cdot C^2 = 25 C^2$ (a **28% reduction** in parameters) and introduce **two non-linear activation functions** instead of one!

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../python/module04_spatial_vision_and_cnns'))
from receptive_field import build_lenet5, build_vgg16_early_stage

print("=== LeNet-5 (1998) Receptive Field Tracking ===")
print(build_lenet5().summary_table())

print("\n=== VGG-16 Early Stage Receptive Field Tracking ===")
print(build_vgg16_early_stage().summary_table())

## 3. The 2015 Deep Degradation Problem & The ResNet Highway (He et al., 2015)

Before ResNet, researchers assumed deeper networks always performed better until overfitting set in. But in 2015, Kaiming He et al. discovered an astonishing paradox:
> A 56-layer plain network suffered higher **training error** than a 20-layer plain network!

Because training error was higher, this was NOT overfitting. And because BatchNorm was present, gradients were not vanishing. The cause was **Optimization Degradation**: SGD simply cannot optimize deep stacks of non-linear transformations to learn the identity function $\mathcal{H}(x) = x$.

### The Residual Solution:
$$\mathcal{H}(x) = \mathcal{F}(x) + x$$
By adding the skip connection $x$, the gradient flows backward as:
$$\frac{\partial \mathcal{E}}{\partial x_l} = \frac{\partial \mathcal{E}}{\partial x_L} \left( \mathbf{I} + \frac{\partial}{\partial x_l} \sum_{i=l}^{L-1} \mathcal{F}(x_i) \right)$$
The identity term $\mathbf{I}$ guarantees that gradient signal flows directly from the loss to the first layer, completely unaffected by vanishing weights!

In [ ]:
from resnet_pytorch import run_degradation_experiment

print("Running Deep Degradation Experiment (30-layer PlainNet vs 30-layer ResNet)...\n")
results = run_degradation_experiment(num_blocks=14, channels=16, steps=25)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(results['plain_losses'], label='30-layer PlainNet (No Skips)', color='#f43f5e', linewidth=2)
plt.plot(results['res_losses'], label='30-layer ResNet (With Skips)', color='#10b981', linewidth=2)
plt.title('Training Loss Progression')
plt.xlabel('Optimization Steps')
plt.ylabel('Cross Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(results['plain_grad_norms'], label='PlainNet Layer 1 Gradient Norm', color='#f43f5e', linewidth=2)
plt.plot(results['res_grad_norms'], label='ResNet Layer 1 Gradient Norm', color='#10b981', linewidth=2)
plt.title('First Layer Gradient Norm ||dL/dw_1||')
plt.xlabel('Optimization Steps')
plt.ylabel('Gradient Norm')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final PlainNet Loss: {results['plain_final_loss']:.4f}")
print(f"Final ResNet Loss:   {results['res_final_loss']:.4f} (Dramatically superior optimization!)")

## 4. ConvNeXt: Modernizing CNNs for the 2020s (Liu et al., 2022)

In 2020, Vision Transformers (ViT) threatened to render CNNs obsolete. In 2022, Liu et al. re-engineered ResNet using modern architectural choices borrowed from Transformers:

1. **Macro design**: Changed stage compute ratio to 1:1:3:1 (like Swin Transformer).
2. **Depthwise Separable Convolutions**: Using $7 \times 7$ depthwise convs (matching the larger window size of ViT patches).
3. **Inverted Bottleneck**: Channel expansion $D \to 4D \to D$.
4. **LayerNorm instead of BatchNorm**: Standardizing on feature normalization.
5. **Fewer Activations & Normalizations**: Only 1 GELU and 1 LayerNorm per block.

In [ ]:
import torch
from resnet_pytorch import ConvNeXtBlock, ClassicResidualBlock

x = torch.randn(4, 64, 32, 32)
resnet_block = ClassicResidualBlock(channels=64)
convnext_block = ConvNeXtBlock(dim=64)

out_res = resnet_block(x)
out_next = convnext_block(x)

print("Input Tensor shape:     ", x.shape)
print("ResNet Block output:    ", out_res.shape)
print("ConvNeXt Block output:  ", out_next.shape)
print("\nConvNeXt parameters:")
for name, p in convnext_block.named_parameters():
    print(f"  {name:<20} {tuple(p.shape)}")

print("\nSuccess! Both Module 03 and Module 04 demonstrate the deep evolutionary path of modern deep vision.")